# P13 — Probabilité d'arrêté de catastrophe naturelle par commune

**Contexte métier.** Assureurs (prime Cat Nat), préfet, collectivités : anticiper les communes à risque d'arrêté (indemnisation des sinistres) à partir de l'historique GASPAR et des caractéristiques communales. Enjeu : 247 140 arrêtés depuis 1982, sécheresse en explosion.

**Objectif de l'EDA.** Identifier la cible (arrêté pour une commune/année/risque) et la structure des risques.

**Données.** Base GASPAR — arrêtés de catastrophe naturelle (Géorisques, fichier GASPAR catnat). Licence ouverte. 247 140 arrêtés touchant 34 699 communes, 1982-2025.

## 1. Téléchargement

Archive GASPAR de Géorisques (le zip contient plusieurs fichiers, dont les arrêtés catnat) :

In [ ]:
import urllib.request, zipfile, os, glob
os.makedirs("data", exist_ok=True)

URL = "http://files.georisques.fr/GASPAR/gaspar.zip"
if not os.path.exists("data/catnat_gaspar.csv"):
    urllib.request.urlretrieve(URL, "data/gaspar.zip")
    with zipfile.ZipFile("data/gaspar.zip") as z:
        for n in z.namelist():
            if n.startswith("catnat"):
                z.extract(n, "data")
                os.rename("data/" + n, "data/catnat_gaspar.csv")
                print("extrait :", n)

## 2. Chargement

Fichier des arrêtés catnat : chaque ligne est un arrêté signé pour un couple (commune, risque).

In [ ]:
import pandas as pd, numpy as np

df = pd.read_csv("data/catnat_gaspar.csv", sep=";", low_memory=False, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]
print(len(df), "arretes")

## 3. Analyse

Distribution des risques, évolution annuelle (effet sécheresse), communes les plus récurrentes et nombre de risques distincts par commune.

In [ ]:
df["annee"] = pd.to_datetime(df["date_debut"], errors="coerce").dt.year
print("Risques :", df["lib_risque_jo"].value_counts().head(8).to_dict())
print()
par_an = df.groupby("annee").size()
print("Arrêtés par année (extrait) :", par_an.tail(10).to_dict())
print()
g = df.groupby("code_commune").size()
print("Arrêtés par commune :", g.describe(percentiles=[.5, .75, .9, .99]).round(2).to_dict())
print("Communes avec 5+ arrêtés :", round((g >= 5).mean() * 100, 1), "%")
nr = df.groupby("code_commune")["lib_risque_jo"].nunique()
print("Risques distincts par commune :", nr.describe(percentiles=[.5, .9]).round(2).to_dict())

## 4. Résultats observés (exécution du 16/10/2026)

- n arretes : 247140
- n communes touchees : 34699
- periode : 1982 → 2025
- risques dist : Inondations et/ou Coulées de Boue : 139419, Sécheresse : 43417, Mouvement de Terrain : 31711, Tempête : 16025, Chocs …
- arretes par annee : 1982 : 32903, 1983 : 16679, 1984 : 1823, 1985 : 585, 1986 : 1728, 1987 : 3567, 1988 : 3007, 1989 : 4828, 1990 : 3159,…
- n arretes par dept : 33 : 7816, 31 : 6563, 62 : 5837, 64 : 5793, 11 : 5630, 24 : 5593, 57 : 5449, 17 : 5409, 71 : 4825, 76 : 4448
- communes les plus touchees : Sainte-Marie : 92, Saint-Michel : 86, Nice : 83, Le Pin : 75, Saint-Sauveur : 75, Mons : 72, Sainte-Colombe : 72, Bea…
- n arretes par commune stats : count : 34 699.00, mean : 7.12, std : 4.62, min : 1.00, 50% : 6.00, 75% : 9.00, 90% : 13.00, 99% : 22.00, max : 83.00
- pct communes recurrentes 5plus : 66.64
- n risques distincts par commune stats : count : 34 699.00, mean : 3.01, std : 1.01, min : 1.00, 50% : 3.00, 75% : 4.00, 90% : 4.00, max : 8.00
- annees recentes top risques : Sécheresse : 11288, Inondations et/ou Coulées de Boue : 10040, Mouvement de Terrain : 511, Inondations Remontée Nappe…

## 5. Variable cible et régresseurs

**Cible (classification / dénombrement) : probabilité qu'une commune (x risque x année) fasse l'objet d'un arrêté.** Concrètement : construire une table commune × risque × année et prédire la classe "arrêté".

**Régresseurs (à joindre — GASPAR seul ne contient que les événements) :**
- Risque (`lib_risque_jo`) et historique communal (nombre d'arrêtés passés, récurrence : 4 % des communes ont 5+ arrêtés)
- Département / région (exposition géographique : inondations 56 % des arrêtés)
- Données météo (sécheresses 2022+ : à joindre depuis Météo-France), altimétrie/ZRR,Population (INSEE)
- `annee` (tendance : la sécheresse explose depuis 2018)

**Pièges :**
- GASPAR ne contient que les communes SINISTRÉES : la table d'entraînement exige des contre-exemples (communes × années sans arrêté) — c'est le vrai travail de construction du dataset
- Biais de déclaration : un arrêté dépend aussi de la demande du maire, pas seulement du phénomène
- Dates en mixed format, `code_commune` parfois avec des valeurs spéciales (DOM, arrêtés multi) : normaliser les codes INSEE
- Une année récente incomplète (arrêtés en cours de signature) : ne pas interpréter comme une baisse du risque